# VisClick — Phase 4.4 / D-02 (step 2 of 2): few-shot fine-tune the SSP-adapted detector

**Goal.** Take the SimSiam-adapted backbone from `11_ssp_pretrain.ipynb`, plug it into a fresh YOLOv8s detector (with the source-trained neck and head), then few-shot fine-tune on the hand-corrected desktop set. Compare against the no-SSP curve from `08c_few_shot_curve.ipynb` at the same `k` values.

**Pipeline:**
1. Mount Drive → `git pull` → install.
2. Load `best_source_v8s.pt`. Surgery: replace the backbone weights with the SSP-adapted ones from `<DRIVE>/weights/ssp/backbone_simsiam.pt`.
3. Save the patched checkpoint as `<DRIVE>/weights/ssp/ssp_init_v8s.pt`.
4. For each `k ∈ {1, 8}` (representative low and high points of the curve): head-only fine-tune from the patched checkpoint on the first `k` sorted hand-corrected images. Mirrors the protocol in `08c_few_shot_curve.ipynb`.
5. Evaluate **CPV on ScreenSpot** (n=334, primary) and mAP@0.5 / mAP@0.5:0.95 on the hand-corrected set (secondary, `fit_to_train`).
6. Write `reports/tables/ssp_few_shot.csv` with rows for k=1 and k=8, with and without SSP, side-by-side.
7. Publish.

**Comparison.** Both the no-SSP rows (from 08c) and the SSP rows (from this notebook) are written, so the SSP-vs-no-SSP gap is one subtraction the reader can do without leaving the table.

**Compute reality.** Per-k fine-tune is ~5-10 min on T4. ScreenSpot CPV is ~5 min per checkpoint. Total ~30-45 min for k=1+k=8.

**Report.** Every step prints `REPORT ...` lines.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(
    ["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
import sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics", "datasets", "pillow", "opencv-python", "matplotlib"],
    check=False,
)
import torch, ultralytics
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("ultralytics:", ultralytics.__version__)


## 12.1 — Bootstrap source weights and the SSP-adapted backbone

We need:
- `best_source_v8s.pt` (from `05_train_source.ipynb`) — provides the neck and head.
- `backbone_simsiam.pt` (from `11_ssp_pretrain.ipynb`) — provides the adapted backbone.

If either is missing, the notebook aborts with a clear message.


In [ ]:
import os, shutil, zipfile

DRIVE        = "/content/drive/MyDrive/visclick"
SOURCE_WTS   = os.path.join(DRIVE, "weights", "baseline_source", "best_source_v8s.pt")
SSP_BACKBONE = os.path.join(DRIVE, "weights", "ssp", "backbone_simsiam.pt")
SSP_INIT     = os.path.join(DRIVE, "weights", "ssp", "ssp_init_v8s.pt")
FEWSHOT_DIR  = os.path.join(DRIVE, "weights", "ssp_few_shot")
REPORTS_TBL  = os.path.join(DRIVE, "reports", "tables")
os.makedirs(FEWSHOT_DIR, exist_ok=True)
os.makedirs(REPORTS_TBL, exist_ok=True)

assert os.path.isfile(SOURCE_WTS), f"Missing source weights: {SOURCE_WTS}"
assert os.path.isfile(SSP_BACKBONE), f"Missing SSP backbone: {SSP_BACKBONE}. Run 11_ssp_pretrain first."
print(f"REPORT source_weights | path = {SOURCE_WTS} | size_mb = {os.path.getsize(SOURCE_WTS)/1024/1024:0.1f}")
print(f"REPORT ssp_backbone   | path = {SSP_BACKBONE} | size_mb = {os.path.getsize(SSP_BACKBONE)/1024/1024:0.1f}")

HC_ZIP  = "/content/visclick/datasets/handcorrected_desktop_test/visclick3.yolov8.zip"
HC_ROOT = "/content/hc"
HC_IMG  = os.path.join(HC_ROOT, "train", "images")
HC_LBL  = os.path.join(HC_ROOT, "train", "labels")
if not os.path.isdir(HC_IMG):
    with zipfile.ZipFile(HC_ZIP, "r") as zf:
        zf.extractall(HC_ROOT)
    print("unzipped hand-corrected ->", HC_ROOT)

STEMS = sorted(
    os.path.splitext(f)[0]
    for f in os.listdir(HC_IMG)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
)
N_HC = len(STEMS)
print(f"REPORT hc_pool | n = {N_HC}")

CLASSES = ["button", "text", "text_input", "icon", "menu", "checkbox"]


## 12.2 — Surgery: graft the SSP backbone onto the source-trained detector

Load `best_source_v8s.pt` into an Ultralytics `YOLO`. Replace the first `N_BACKBONE = 10` modules' state dict with the SSP-adapted backbone. Save the patched checkpoint as `ssp_init_v8s.pt`. From here on, the patched checkpoint is what the few-shot loop starts from.


In [ ]:
import torch
from ultralytics import YOLO

src = YOLO(SOURCE_WTS)
det_model = src.model

N_BACKBONE = 10
bb_state = torch.load(SSP_BACKBONE, map_location="cpu")

# Map SimSiam-saved state dict back into the YOLOv8 backbone modules.
target = torch.nn.Sequential(*list(det_model.model[:N_BACKBONE]))
missing, unexpected = target.load_state_dict(bb_state, strict=False)
print(f"REPORT graft | missing_keys = {len(missing)} | unexpected_keys = {len(unexpected)}")
if missing:
    print("  first missing:", missing[:3])
if unexpected:
    print("  first unexpected:", unexpected[:3])

# Persist the patched checkpoint.
src.save(SSP_INIT)
print(f"REPORT ssp_init | path = {SSP_INIT} | size_mb = {os.path.getsize(SSP_INIT)/1024/1024:0.1f}")


## 12.3 — Per-`k` head-only fine-tune from the SSP-initialised checkpoint

Mirrors `08c_few_shot_curve.ipynb` section 8.1: build a `k`-image YOLO dataset (first `k` sorted stems), head-only fine-tune for 30 epochs, save weights to `<DRIVE>/weights/ssp_few_shot/k{k}/`. Resume-aware: if the output `last.pt` exists the cell skips.


In [ ]:
import os, time, yaml, shutil

K_VALUES = [1, 8]
EPOCHS = 30
IMGSZ = 640


def build_yolo_yaml(k: int) -> str:
    work = f"/content/ssp_train_k{k}"
    img_dir = os.path.join(work, "images", "train")
    lbl_dir = os.path.join(work, "labels", "train")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    for stem in STEMS[:k]:
        for ext in (".png", ".jpg", ".jpeg"):
            src_img = os.path.join(HC_IMG, stem + ext)
            if os.path.isfile(src_img):
                shutil.copy2(src_img, os.path.join(img_dir, os.path.basename(src_img)))
                break
        src_lbl = os.path.join(HC_LBL, stem + ".txt")
        if os.path.isfile(src_lbl):
            shutil.copy2(src_lbl, os.path.join(lbl_dir, stem + ".txt"))
    yaml_path = os.path.join(work, "data.yaml")
    with open(yaml_path, "w") as fh:
        yaml.safe_dump({
            "path": work, "train": "images/train", "val": "images/train",
            "names": CLASSES, "nc": len(CLASSES),
        }, fh)
    return yaml_path


for k in K_VALUES:
    out_dir = os.path.join(FEWSHOT_DIR, f"k{k}")
    last_pt = os.path.join(out_dir, "weights", "last.pt")
    if os.path.isfile(last_pt):
        print(f"k={k}: weights already at {last_pt}; skipping.")
        continue

    data_yaml = build_yolo_yaml(k)
    t0 = time.time()
    model = YOLO(SSP_INIT)
    model.train(
        data=data_yaml,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=4,
        project=FEWSHOT_DIR,
        name=f"k{k}",
        freeze=10,
        verbose=False,
        plots=False,
    )
    print(f"REPORT ssp_finetune | k = {k} | epochs = {EPOCHS} | elapsed = {time.time()-t0:0.1f}s")


## 12.4 — Evaluate: ScreenSpot CPV (primary) + hand-corrected mAP (secondary)

Reuses the proven evaluation scripts already in the repo:
- `scripts/run_cpv_screenspot.py` — instruction-grounded CPV on the ScreenSpot desktop slice (n=334).
- `scripts/run_cpv.py` — per-element-recall CPV on the hand-corrected set (`fit_to_train` for k>0).

The ScreenSpot CPV is the headline number reported in Section 7.x. The hand-corrected CPV is the supplementary fit-to-train sanity check.


In [ ]:
import subprocess, tempfile

per_k_results = {}

for k in K_VALUES:
    weights = os.path.join(FEWSHOT_DIR, f"k{k}", "weights", "best.pt")
    if not os.path.isfile(weights):
        weights = os.path.join(FEWSHOT_DIR, f"k{k}", "weights", "last.pt")
    assert os.path.isfile(weights), f"No checkpoint for k={k}"

    # Export ONNX for the CPV scripts (they accept .onnx or .pt; .pt is fine).
    onnx_out = weights.replace(".pt", ".onnx")
    if not os.path.isfile(onnx_out):
        YOLO(weights).export(format="onnx", imgsz=IMGSZ, dynamic=False, opset=12)

    with tempfile.TemporaryDirectory() as tmp:
        ss_csv = os.path.join(tmp, f"ss_k{k}.csv")
        subprocess.run([
            sys.executable, "/content/visclick/scripts/run_cpv_screenspot.py",
            "--weights", onnx_out, "--out", ss_csv,
        ], check=True)
        with open(ss_csv) as fh:
            head = next(fh); ss_overall = next(fh).strip().split(",")
        cpv_ss = float(ss_overall[-1])

        hc_csv = os.path.join(tmp, f"hc_k{k}.csv")
        subprocess.run([
            sys.executable, "/content/visclick/scripts/run_cpv.py",
            "--weights", onnx_out, "--out", hc_csv,
        ], check=True)
        with open(hc_csv) as fh:
            head = next(fh)
            hc_overall = None
            for line in fh:
                parts = line.strip().split(",")
                if parts and parts[0] == "OVERALL":
                    hc_overall = parts
                    break
        cpv_hc = float(hc_overall[-1]) if hc_overall else float("nan")

    per_k_results[k] = {"cpv_screenspot_%": cpv_ss, "cpv_handcorrected_%": cpv_hc}
    print(f"REPORT ssp_eval | k = {k} | cpv_screenspot = {cpv_ss:.2f} | cpv_handcorrected = {cpv_hc:.2f}")


## 12.5 — Write `ssp_few_shot.csv` with both SSP and no-SSP rows

Reads the no-SSP numbers from `reports/tables/sample_efficiency.csv` (produced by 08c) and writes a side-by-side comparison. The reader can compute the SSP gap by subtracting columns.


In [ ]:
import csv

NO_SSP_CSV = "/content/visclick/reports/tables/sample_efficiency.csv"
no_ssp = {}
if os.path.isfile(NO_SSP_CSV):
    with open(NO_SSP_CSV) as fh:
        for row in csv.DictReader(fh):
            try:
                k = int(row["k"])
                no_ssp[k] = row
            except (KeyError, ValueError):
                continue
else:
    print(f"WARN: no baseline at {NO_SSP_CSV}; SSP rows written without comparison.")

OUT_CSV = "/content/visclick/reports/tables/ssp_few_shot.csv"
with open(OUT_CSV, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["k", "method", "cpv_screenspot_%", "cpv_handcorrected_%"])
    for k in K_VALUES:
        if k in no_ssp:
            r = no_ssp[k]
            w.writerow([k, "no-ssp",
                        r.get("cpv_screenspot_%", ""),
                        r.get("cpv_handcorrected_%", "")])
        d = per_k_results[k]
        w.writerow([k, "ssp",
                    f"{d['cpv_screenspot_%']:.2f}",
                    f"{d['cpv_handcorrected_%']:.2f}"])

print(f"REPORT step = WRITE_CSV | path = {OUT_CSV}")
shutil.copy2(OUT_CSV, os.path.join(REPORTS_TBL, "ssp_few_shot.csv"))


## 12.6 — Publish to git

In [ ]:
import os, subprocess

REPO_ROOT = "/content/visclick"
ARTIFACTS = [
    'reports/tables/ssp_few_shot.csv',
]

for rel in ARTIFACTS:
    p = os.path.join(REPO_ROOT, rel)
    assert os.path.exists(p), f"Missing artifact in repo clone: {p}. Run the previous section first."
    print(f"OK  {p}  ({os.path.getsize(p)} bytes)")

token_path = os.path.join(REPO_ROOT, "token")
if not os.path.exists(token_path):
    print(f"WARN: no token file at {token_path}; skipping git push. Copy artifacts in by hand or restore the token.")
else:
    with open(token_path) as fh:
        token = fh.read().strip()

    def run(cmd, **kw):
        r = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True, **kw)
        if r.returncode != 0:
            print("STDOUT:", r.stdout)
            print("STDERR:", r.stderr)
            raise RuntimeError(f"git command failed: {' '.join(cmd)}")
        return r.stdout

    run(["git", "config", "user.email", "hiran@iit.ac.lk"])
    run(["git", "config", "user.name",  "Hiran Abeywardhana"])

    run(["git", "add", *ARTIFACTS])

    status = run(["git", "status", "--porcelain"])
    if not status.strip():
        print("REPORT step = GIT_PUBLISH | status = NOTHING_TO_COMMIT")
    else:
        run(["git", "commit", "-m", 'D-02: SSP+FT few-shot results vs no-SSP baseline'])
        url = f"https://{token}@github.com/HiranMadhu/visclick.git"
        push = subprocess.run(["git", "push", url, "HEAD:main"],
                              cwd=REPO_ROOT, capture_output=True, text=True)
        if push.returncode != 0:
            print("PUSH STDERR:", push.stderr)
            raise RuntimeError("git push failed")
        print("REPORT step = GIT_PUBLISH | status = PUSHED")
